# Sidewinder FF2 — Force Feedback Lab

Run cells top to bottom. Each section has sliders that update the effect **live** while it is playing.

**Requirements:** `brew install hidapi` and `pip install -r requirements.txt` — no sudo needed.

**Hold the stick:** the FF2 has a hand sensor and cuts all force output the moment you let go.

In [1]:
import time
import ipywidgets as w
from IPython.display import display
from sidewinder.ff2 import (
    FF2Device,
    ET_CONSTANT_FORCE, ET_RAMP,
    ET_SINE, ET_SQUARE, ET_TRIANGLE, ET_SAWTOOTH_UP, ET_SAWTOOTH_DOWN,
    ET_SPRING, ET_DAMPER, ET_INERTIA, ET_FRICTION,
    ET_NAMES,
)

ff = FF2Device()
print("Connected:", ff)

Connected: <FF2Device VID=0x045e PID=0x001b>


---
## Device Controls

In [2]:
gain_sl = w.IntSlider(value=255, min=0, max=255, step=1,
                      description='Global Gain:', style={'description_width': '120px'},
                      layout=w.Layout(width='400px'))

stop_all_btn = w.Button(description='⏹ Stop All', button_style='warning')
pause_btn    = w.ToggleButton(value=False, description='⏸ Pause', button_style='')

def _gain(change):  ff.set_gain(change['new'])
def _stop_all(_):   ff.stop_all()
def _pause(change): ff.pause() if change['new'] else ff.resume()

gain_sl.observe(_gain, names='value')
stop_all_btn.on_click(_stop_all)
pause_btn.observe(_pause, names='value')

display(w.VBox([
    gain_sl,
    w.HBox([stop_all_btn, pause_btn]),
]))

---
## 1 · Constant Force
Pushes the stick in a fixed direction. Sliders update the effect live while playing.

In [3]:
_cf = [None]

cf_angle = w.IntSlider(value=90,  min=0,   max=360,  step=1,  description='Angle °',    style={'description_width':'100px'}, layout=w.Layout(width='450px'))
cf_mag   = w.IntSlider(value=200, min=0,   max=255,  step=1,  description='Magnitude',  style={'description_width':'100px'}, layout=w.Layout(width='450px'))
cf_gain  = w.IntSlider(value=255, min=0,   max=255,  step=1,  description='Gain',       style={'description_width':'100px'}, layout=w.Layout(width='450px'))
cf_atk   = w.IntSlider(value=0,   min=0,   max=2000, step=50, description='Attack ms',  style={'description_width':'100px'}, layout=w.Layout(width='450px'))
cf_atl   = w.IntSlider(value=0,   min=0,   max=255,  step=1,  description='Attack lvl', style={'description_width':'100px'}, layout=w.Layout(width='450px'))
cf_fade  = w.IntSlider(value=0,   min=0,   max=2000, step=50, description='Fade ms',    style={'description_width':'100px'}, layout=w.Layout(width='450px'))
cf_fadl  = w.IntSlider(value=0,   min=0,   max=255,  step=1,  description='Fade lvl',   style={'description_width':'100px'}, layout=w.Layout(width='450px'))

cf_play  = w.ToggleButton(value=False, description='▶ Play', button_style='success', layout=w.Layout(width='90px'))
cf_stat  = w.Label('—')

def _cf_apply():
    e = _cf[0]
    if e is None: return
    e.set_base(direction_deg=cf_angle.value, gain=cf_gain.value,
               direction_enable=True, axis_x=True, axis_y=True)
    e.set_constant_force(cf_mag.value)

def _cf_toggle(change):
    if change['new']:
        if _cf[0]: _cf[0].free()
        e = ff.alloc(ET_CONSTANT_FORCE)
        _cf[0] = e
        e.set_base(direction_deg=cf_angle.value, gain=cf_gain.value,
                   direction_enable=True, axis_x=True, axis_y=True)
        if cf_atk.value or cf_fade.value:
            e.set_envelope(attack_level=cf_atl.value, attack_ms=cf_atk.value,
                           fade_level=cf_fadl.value, fade_ms=cf_fade.value)
        e.set_constant_force(cf_mag.value)
        e.play()
        cf_play.description = '⏹ Stop'; cf_play.button_style = 'danger'
        cf_stat.value = f'block {e.block}'
    else:
        if _cf[0]: _cf[0].free(); _cf[0] = None
        cf_play.description = '▶ Play'; cf_play.button_style = 'success'
        cf_stat.value = '—'

cf_play.observe(_cf_toggle, names='value')
for s in [cf_angle, cf_mag, cf_gain]:
    s.observe(lambda _: _cf_apply(), names='value')

display(w.VBox([
    cf_angle, cf_mag, cf_gain,
    w.HTML('<details><summary style="cursor:pointer;color:#888">Envelope (attack / fade)</summary>'),
    cf_atk, cf_atl, cf_fade, cf_fadl,
    w.HTML('</details>'),
    w.HBox([cf_play, cf_stat]),
]))

---
## 2 · Periodic Effects
Oscillating force: Sine, Square, Triangle, Sawtooth Up/Down.

In [3]:
_pe = [None]

_PERIODIC_OPTS = [
    ('Sine',          ET_SINE),
    ('Square',        ET_SQUARE),
    ('Triangle',      ET_TRIANGLE),
    ('Sawtooth Up',   ET_SAWTOOTH_UP),
    ('Sawtooth Down', ET_SAWTOOTH_DOWN),
]

pe_type   = w.Dropdown(options=_PERIODIC_OPTS, value=ET_SINE, description='Waveform:', style={'description_width':'100px'})
pe_angle  = w.IntSlider(value=90,  min=0,    max=360,  step=1,  description='Angle °',   style={'description_width':'100px'}, layout=w.Layout(width='450px'))
pe_mag    = w.IntSlider(value=255, min=0,    max=255,  step=1,  description='Magnitude', style={'description_width':'100px'}, layout=w.Layout(width='450px'))
pe_offset = w.IntSlider(value=0,   min=-128, max=127,  step=1,  description='Offset',    style={'description_width':'100px'}, layout=w.Layout(width='450px'))
pe_phase  = w.IntSlider(value=0,   min=0,    max=360,  step=1,  description='Phase °',   style={'description_width':'100px'}, layout=w.Layout(width='450px'))
pe_period = w.IntSlider(value=500, min=20,   max=5000, step=20, description='Period ms', style={'description_width':'100px'}, layout=w.Layout(width='450px'))
pe_gain   = w.IntSlider(value=255, min=0,    max=255,  step=1,  description='Gain',      style={'description_width':'100px'}, layout=w.Layout(width='450px'))
pe_atk    = w.IntSlider(value=0,   min=0,    max=2000, step=50, description='Attack ms', style={'description_width':'100px'}, layout=w.Layout(width='450px'))
pe_fade   = w.IntSlider(value=0,   min=0,    max=2000, step=50, description='Fade ms',   style={'description_width':'100px'}, layout=w.Layout(width='450px'))

pe_play   = w.ToggleButton(value=False, description='▶ Play', button_style='success', layout=w.Layout(width='90px'))
pe_stat   = w.Label('—')

def _pe_apply():
    e = _pe[0]
    if e is None: return
    e.set_base(direction_deg=pe_angle.value, gain=pe_gain.value,
               direction_enable=True, axis_x=True, axis_y=True)
    e.set_periodic(magnitude=pe_mag.value, offset=pe_offset.value,
                   phase_deg=pe_phase.value, period_ms=pe_period.value)

def _pe_restart():
    if _pe[0]: _pe[0].free()
    e = ff.alloc(pe_type.value)
    _pe[0] = e
    e.set_base(direction_deg=pe_angle.value, gain=pe_gain.value,
               direction_enable=True, axis_x=True, axis_y=True)
    if pe_atk.value or pe_fade.value:
        e.set_envelope(attack_ms=pe_atk.value, fade_ms=pe_fade.value)
    e.set_periodic(magnitude=pe_mag.value, offset=pe_offset.value,
                   phase_deg=pe_phase.value, period_ms=pe_period.value)
    e.play()
    pe_stat.value = f'block {e.block}'

def _pe_toggle(change):
    if change['new']:
        _pe_restart()
        pe_play.description = '⏹ Stop'; pe_play.button_style = 'danger'
    else:
        if _pe[0]: _pe[0].free(); _pe[0] = None
        pe_play.description = '▶ Play'; pe_play.button_style = 'success'
        pe_stat.value = '—'

pe_play.observe(_pe_toggle, names='value')
for s in [pe_angle, pe_mag, pe_offset, pe_phase, pe_period, pe_gain]:
    s.observe(lambda _: _pe_apply(), names='value')
# Waveform change requires a new alloc (different effect type)
pe_type.observe(lambda _: _pe_restart() if pe_play.value else None, names='value')

display(w.VBox([
    pe_type, pe_angle, pe_mag, pe_offset, pe_phase, pe_period, pe_gain,
    w.HTML('<details><summary style="cursor:pointer;color:#888">Envelope (attack / fade)</summary>'),
    pe_atk, pe_fade,
    w.HTML('</details>'),
    w.HBox([pe_play, pe_stat]),
]))

---
## 3 · Spring
Restoring force toward a center point. Good for simulating a physical spring or trim.

In [4]:
_sp = [None]

sp_stiff  = w.IntSlider(value=100, min=0,    max=127, step=1, description='Stiffness',  style={'description_width':'120px'}, layout=w.Layout(width='450px'))
sp_cpx    = w.IntSlider(value=0,   min=-127, max=127, step=1, description='Center X',   style={'description_width':'120px'}, layout=w.Layout(width='450px'))
sp_cpy    = w.IntSlider(value=0,   min=-127, max=127, step=1, description='Center Y',   style={'description_width':'120px'}, layout=w.Layout(width='450px'))
sp_sat    = w.IntSlider(value=255, min=0,    max=255, step=1, description='Saturation', style={'description_width':'120px'}, layout=w.Layout(width='450px'))
sp_dead   = w.IntSlider(value=0,   min=0,    max=255, step=1, description='Dead band',  style={'description_width':'120px'}, layout=w.Layout(width='450px'))
sp_gain   = w.IntSlider(value=255, min=0,    max=255, step=1, description='Gain',       style={'description_width':'120px'}, layout=w.Layout(width='450px'))

sp_play   = w.ToggleButton(value=False, description='▶ Play', button_style='success', layout=w.Layout(width='90px'))
sp_stat   = w.Label('—')

def _sp_apply():
    e = _sp[0]
    if e is None: return
    e.set_base(axis_x=True, axis_y=True, direction_enable=False, gain=sp_gain.value)
    for axis, cp in ((0, sp_cpx.value), (1, sp_cpy.value)):
        e.set_condition(axis=axis, cp_offset=cp,
                        pos_coeff=sp_stiff.value, neg_coeff=sp_stiff.value,
                        pos_sat=sp_sat.value, neg_sat=sp_sat.value,
                        dead_band=sp_dead.value)

def _sp_toggle(change):
    if change['new']:
        if _sp[0]: _sp[0].free()
        e = ff.alloc(ET_SPRING)
        _sp[0] = e
        _sp_apply()
        e.play()
        sp_play.description = '⏹ Stop'; sp_play.button_style = 'danger'
        sp_stat.value = f'block {e.block}'
    else:
        if _sp[0]: _sp[0].free(); _sp[0] = None
        sp_play.description = '▶ Play'; sp_play.button_style = 'success'
        sp_stat.value = '—'

sp_play.observe(_sp_toggle, names='value')
for s in [sp_stiff, sp_cpx, sp_cpy, sp_sat, sp_dead, sp_gain]:
    s.observe(lambda _: _sp_apply(), names='value')

display(w.VBox([sp_stiff, sp_cpx, sp_cpy, sp_sat, sp_dead, sp_gain,
                w.HBox([sp_play, sp_stat])]))

---
## 4 · Damper
Resists stick velocity — higher coefficient = more resistance to movement.

In [5]:
_dm = [None]

dm_coeff = w.IntSlider(value=80,  min=0, max=127, step=1, description='Coefficient', style={'description_width':'120px'}, layout=w.Layout(width='450px'))
dm_sat   = w.IntSlider(value=255, min=0, max=255, step=1, description='Saturation',  style={'description_width':'120px'}, layout=w.Layout(width='450px'))
dm_gain  = w.IntSlider(value=255, min=0, max=255, step=1, description='Gain',        style={'description_width':'120px'}, layout=w.Layout(width='450px'))

dm_play  = w.ToggleButton(value=False, description='▶ Play', button_style='success', layout=w.Layout(width='90px'))
dm_stat  = w.Label('—')

def _dm_apply():
    e = _dm[0]
    if e is None: return
    e.set_base(axis_x=True, axis_y=True, direction_enable=False, gain=dm_gain.value)
    for axis in (0, 1):
        e.set_condition(axis=axis, pos_coeff=dm_coeff.value, neg_coeff=dm_coeff.value,
                        pos_sat=dm_sat.value, neg_sat=dm_sat.value)

def _dm_toggle(change):
    if change['new']:
        if _dm[0]: _dm[0].free()
        e = ff.alloc(ET_DAMPER); _dm[0] = e
        _dm_apply(); e.play()
        dm_play.description = '⏹ Stop'; dm_play.button_style = 'danger'
        dm_stat.value = f'block {e.block}'
    else:
        if _dm[0]: _dm[0].free(); _dm[0] = None
        dm_play.description = '▶ Play'; dm_play.button_style = 'success'
        dm_stat.value = '—'

dm_play.observe(_dm_toggle, names='value')
for s in [dm_coeff, dm_sat, dm_gain]:
    s.observe(lambda _: _dm_apply(), names='value')

display(w.VBox([dm_coeff, dm_sat, dm_gain, w.HBox([dm_play, dm_stat])]))

---
## 5 · Friction
Constant opposing force — activates as soon as stick leaves the dead band.

In [6]:
_fr = [None]

fr_level = w.IntSlider(value=80, min=0, max=127, step=1, description='Level',     style={'description_width':'120px'}, layout=w.Layout(width='450px'))
fr_dead  = w.IntSlider(value=10, min=0, max=255, step=1, description='Dead band', style={'description_width':'120px'}, layout=w.Layout(width='450px'))
fr_gain  = w.IntSlider(value=255,min=0, max=255, step=1, description='Gain',      style={'description_width':'120px'}, layout=w.Layout(width='450px'))

fr_play  = w.ToggleButton(value=False, description='▶ Play', button_style='success', layout=w.Layout(width='90px'))
fr_stat  = w.Label('—')

def _fr_apply():
    e = _fr[0]
    if e is None: return
    e.set_base(axis_x=True, axis_y=True, direction_enable=False, gain=fr_gain.value)
    for axis in (0, 1):
        e.set_condition(axis=axis, pos_coeff=fr_level.value, neg_coeff=fr_level.value,
                        dead_band=fr_dead.value)

def _fr_toggle(change):
    if change['new']:
        if _fr[0]: _fr[0].free()
        e = ff.alloc(ET_FRICTION); _fr[0] = e
        _fr_apply(); e.play()
        fr_play.description = '⏹ Stop'; fr_play.button_style = 'danger'
        fr_stat.value = f'block {e.block}'
    else:
        if _fr[0]: _fr[0].free(); _fr[0] = None
        fr_play.description = '▶ Play'; fr_play.button_style = 'success'
        fr_stat.value = '—'

fr_play.observe(_fr_toggle, names='value')
for s in [fr_level, fr_dead, fr_gain]:
    s.observe(lambda _: _fr_apply(), names='value')

display(w.VBox([fr_level, fr_dead, fr_gain, w.HBox([fr_play, fr_stat])]))

---
## 6 · Inertia
Resists acceleration — stick feels heavy, harder to start and stop moving.

In [7]:
_in = [None]

in_coeff = w.IntSlider(value=80,  min=0, max=127, step=1, description='Coefficient', style={'description_width':'120px'}, layout=w.Layout(width='450px'))
in_sat   = w.IntSlider(value=255, min=0, max=255, step=1, description='Saturation',  style={'description_width':'120px'}, layout=w.Layout(width='450px'))
in_gain  = w.IntSlider(value=255, min=0, max=255, step=1, description='Gain',        style={'description_width':'120px'}, layout=w.Layout(width='450px'))

in_play  = w.ToggleButton(value=False, description='▶ Play', button_style='success', layout=w.Layout(width='90px'))
in_stat  = w.Label('—')

def _in_apply():
    e = _in[0]
    if e is None: return
    e.set_base(axis_x=True, axis_y=True, direction_enable=False, gain=in_gain.value)
    for axis in (0, 1):
        e.set_condition(axis=axis, pos_coeff=in_coeff.value, neg_coeff=in_coeff.value,
                        pos_sat=in_sat.value, neg_sat=in_sat.value)

def _in_toggle(change):
    if change['new']:
        if _in[0]: _in[0].free()
        e = ff.alloc(ET_INERTIA); _in[0] = e
        _in_apply(); e.play()
        in_play.description = '⏹ Stop'; in_play.button_style = 'danger'
        in_stat.value = f'block {e.block}'
    else:
        if _in[0]: _in[0].free(); _in[0] = None
        in_play.description = '▶ Play'; in_play.button_style = 'success'
        in_stat.value = '—'

in_play.observe(_in_toggle, names='value')
for s in [in_coeff, in_sat, in_gain]:
    s.observe(lambda _: _in_apply(), names='value')

display(w.VBox([in_coeff, in_sat, in_gain, w.HBox([in_play, in_stat])]))

---
## 7 · Ramp Force
Force linearly ramps from `start` to `end` magnitude over `duration_ms`, then repeats.

In [10]:
_rp = [None]

rp_angle = w.IntSlider(value=90,   min=0,    max=360,   step=1,   description='Angle °',     style={'description_width':'120px'}, layout=w.Layout(width='450px'))
rp_start = w.IntSlider(value=-127, min=-128, max=127,   step=1,   description='Start',       style={'description_width':'120px'}, layout=w.Layout(width='450px'))
rp_end   = w.IntSlider(value=127,  min=-128, max=127,   step=1,   description='End',         style={'description_width':'120px'}, layout=w.Layout(width='450px'))
rp_dur   = w.IntSlider(value=2000, min=100,  max=10000, step=100, description='Duration ms', style={'description_width':'120px'}, layout=w.Layout(width='450px'))
rp_gain  = w.IntSlider(value=255,  min=0,    max=255,   step=1,   description='Gain',        style={'description_width':'120px'}, layout=w.Layout(width='450px'))

rp_play  = w.ToggleButton(value=False, description='▶ Play', button_style='success', layout=w.Layout(width='90px'))
rp_stat  = w.Label('—')

def _rp_restart():
    if _rp[0]: _rp[0].free()
    e = ff.alloc(ET_RAMP)
    _rp[0] = e
    e.set_base(direction_deg=rp_angle.value, gain=rp_gain.value,
               direction_enable=True, axis_x=True, axis_y=True,
               duration_ms=rp_dur.value)
    e.set_ramp_force(rp_start.value, rp_end.value)
    e.play()
    rp_stat.value = f'block {e.block}'

def _rp_toggle(change):
    if change['new']:
        _rp_restart()
        rp_play.description = '⏹ Stop'; rp_play.button_style = 'danger'
    else:
        if _rp[0]: _rp[0].free(); _rp[0] = None
        rp_play.description = '▶ Play'; rp_play.button_style = 'success'
        rp_stat.value = '—'

rp_play.observe(_rp_toggle, names='value')
for s in [rp_angle, rp_start, rp_end, rp_dur, rp_gain]:
    s.observe(lambda _: _rp_restart() if rp_play.value else None, names='value')

display(w.VBox([rp_angle, rp_start, rp_end, rp_dur, rp_gain,
                w.HBox([rp_play, rp_stat])]))

---
## 8 · Combination Example
Spring + Damper simultaneously — simulates a realistic physical stick feel.

Run this cell to start, run again to stop.

In [7]:
_combo = []

if _combo:
    for e in _combo: e.free()
    _combo.clear()
    print("Stopped.")
else:
    _combo.append(ff.spring(stiffness=80, saturation=200))
    _combo.append(ff.damper(coefficient=40, saturation=150))
    print(f"Playing: {_combo}")

Playing: [<Effect block=8 type=Spring playing>, <Effect block=9 type=Damper playing>]


In [8]:
ff.stop_all()

<FF2Device VID=0x045e PID=0x001b>

# Own example

In [21]:
if '_dyn_stop' in vars(): _dyn_stop.set(); time.sleep(0.05)                                                     
if '_pot_effect' in vars() and _pot_effect:                
  try: _pot_effect.free()                                                                                     
  except: pass                                                                                                
_pot_effect = None
if '_attractors' in vars() and _attractors:                                                                     
  for _e in _attractors:                                
      try: _e.free()                                                                                          
      except: pass  
_attractors = []                                                                                                
ff.reset()                                                
ff.enable()
print("Device reset — all blocks cleared.")

ff.stop_all()
print("Standard spring stopped")

Device reset — all blocks cleared.
Standard spring stopped


In [81]:
# import random

# N          = 1      # number of attractors                
# STIFFNESS  = 127     # pos/neg coefficient  (0–127)
# SATURATION = 255    # force cap            (0–255)                                                              
# GAIN       = 220    # overall gain         (0–255)                                                              
                                                                                                              
# _attractors = []                                                                                                
# for i in range(N):                                        
#   cx = random.randint(-110, 110)                                                                              
#   cy = random.randint(-110, 110)                        
#   e = ff.alloc(ET_SPRING)
#   e.set_base(axis_x=True, axis_y=True, direction_enable=False, gain=GAIN)
#   e.set_condition(axis=0, cp_offset=cx,                                                                       
#                   pos_coeff=STIFFNESS, neg_coeff=STIFFNESS,
#                   pos_sat=SATURATION,  neg_sat=SATURATION)                                                    
#   e.set_condition(axis=1, cp_offset=cy,                 
#                   pos_coeff=STIFFNESS, neg_coeff=STIFFNESS,                                                   
#                   pos_sat=SATURATION,  neg_sat=SATURATION)
#   e.play()                                                                                                    
#   _attractors.append(e)                                 
#   print(f"  attractor {i+1:2d}: center=({cx:+4d}, {cy:+4d})  block={e.block}")                                

# print(f"\n{N} attractors active. Re-run cell 1 to clear.")

  attractor  1: center=( -54,  -24)  block=2

1 attractors active. Re-run cell 1 to clear.


# Jumping spring approximation

In [17]:
import io, threading, time, math
import numpy as np                                                                                              
import matplotlib
matplotlib.use('Agg')                                                                                           
import matplotlib.pyplot as plt                           
import matplotlib.patches as mpatches
import ipywidgets as w
from IPython.display import display                                                                             

WELLS = [                                                                                                       
  (-0.55, -0.45,  1.0,  0.22),                          
  ( 0.50, -0.30,  0.9,  0.20),
  ( 0.05,  0.55,  1.0,  0.18),                                                                                
  (-0.20,  0.10,  0.6,  0.15),
  ( 0.60,  0.42,  0.7,  0.20),                                                                                
]                                                                                                               

def V(x, y):                                                                                                    
  v = np.zeros_like(np.asarray(x, float))               
  for cx, cy, A, s in WELLS:
      v -= A * np.exp(-((x - cx)**2 + (y - cy)**2) / (2 * s**2))
  return v                                                                                                    

def dominant_well(x, y):                                                                                        
  best_cx, best_cy, best_f = 0.0, 0.0, -1.0             
  for cx, cy, A, s in WELLS:                                                                                  
      r2 = (x - cx)**2 + (y - cy)**2
      f  = (A / s**2) * math.exp(-r2 / (2 * s**2))                                                            
      if f > best_f:                                                                                          
          best_f = f
          best_cx, best_cy = cx, cy                                                                           
  return best_cx, best_cy                               

# ── Build static background once ─────────────────────────────────────────────                                 
_res = 250
_xg  = np.linspace(-1, 1, _res)                                                                                 
_yg  = np.linspace(-1, 1, _res)                           
_XX, _YY = np.meshgrid(_xg, _yg)                                                                                
_ZZ = V(_XX, _YY)
                                                                                                              
fig, ax = plt.subplots(figsize=(5, 5), dpi=300)                                                                  
ax.contourf(_XX, _YY, _ZZ, levels=50, cmap='RdYlBu_r')
ax.contour (_XX, _YY, _ZZ, levels=20, colors='k', linewidths=0.3, alpha=0.3)                                    
for cx, cy, A, s in WELLS:                                                                                      
  ax.add_patch(mpatches.Circle((cx, cy), s, color='white', fill=False,
                                lw=1.2, ls='--', alpha=0.5, zorder=3))                                        
  ax.plot(cx, cy, 'w+', ms=11, mew=2, zorder=3)         
ax.set_xlim(-1, 1); ax.set_ylim(-1, 1) 
ax.invert_yaxis()
ax.set_xlabel('X'); ax.set_ylabel('Y')                    
ax.set_title('Potential field — live')                                                                          
fig.tight_layout()                                                                                              

# Capture background (no dynamic elements yet)                                                                  
_bg_buf = io.BytesIO()                                    
fig.savefig(_bg_buf, format='png')                                                                              
_bg_bytes = _bg_buf.getvalue()                            
                                                                                                              
# Dynamic artists (added on top each frame)                                                                     
_active_ring = mpatches.Circle((0, 0), 0.01, color='yellow', fill=False,                                        
                               lw=3, alpha=0.9, zorder=4)                                                     
ax.add_patch(_active_ring)                                                                                      
_spring_line, = ax.plot([], [], color='yellow', lw=1.5, alpha=0.7, zorder=4)
_stick_dot,   = ax.plot([0], [0], 'o', color='red', ms=9, zorder=5)                                             
                                                                                                              
def _render():
  buf = io.BytesIO()                                                                                          
  fig.savefig(buf, format='png')                                                                              
  return buf.getvalue()
                                                                                                              
# Widget to display into                                  
_img_widget = w.Image(value=_bg_bytes, format='png')
_info_label = w.Label('starting…')                                                                              
display(w.VBox([_img_widget, _info_label]))                                                                     
                                                                                                              
# ── Spring FF + live update loop ──────────────────────────────────────────────                                
STIFFNESS  = 255                                          
SATURATION = 150                                                                                                
GAIN       = 255                                          

if '_dyn_stop' in vars():
  _dyn_stop.set(); time.sleep(0.05)
if '_pot_effect' in vars() and _pot_effect:                                                                     
  try: _pot_effect.free()
  except: pass                                                                                                
                                                        
_pot_effect = ff.alloc(ET_SPRING)
_pot_effect.set_base(axis_x=True, axis_y=True, direction_enable=False, gain=GAIN)
_pot_effect.set_condition(axis=0, cp_offset=0,                                                                  
                        pos_coeff=STIFFNESS, neg_coeff=STIFFNESS,
                        pos_sat=SATURATION,  neg_sat=SATURATION)                                              
_pot_effect.set_condition(axis=1, cp_offset=0,            
                        pos_coeff=STIFFNESS, neg_coeff=STIFFNESS,                                             
                        pos_sat=SATURATION,  neg_sat=SATURATION)                                              
_pot_effect.play()
                                                                                                              
_dyn_stop = threading.Event()                                                                                   
                                                                                                              
def _pot_loop():                                                                                                
  prev_well = (None, None)                              
  pos       = (0.0, 0.0)   # last known good position                                                         
  frame     = 0
  while not _dyn_stop.is_set():                                                                               
      try:                                              
          result = ff.read_position()                                                                         
          if result is not None:
              pos = result                                                                                    
          x, y   = pos                                  
          cx, cy = dominant_well(x, y)

          if (cx, cy) != prev_well:                                                                           
              cpx, cpy = int(cx * 127), int(cy * 127)
              _pot_effect.set_condition(axis=0, cp_offset=cpx,                                                
                                        pos_coeff=STIFFNESS, neg_coeff=STIFFNESS,
                                        pos_sat=SATURATION,  neg_sat=SATURATION)                              
              _pot_effect.set_condition(axis=1, cp_offset=cpy,
                                        pos_coeff=STIFFNESS, neg_coeff=STIFFNESS,                             
                                        pos_sat=SATURATION,  neg_sat=SATURATION)                              
              sigma = next(s for (wcx, wcy, A, s) in WELLS
                           if wcx == cx and wcy == cy)                                                        
              _active_ring.set_center((cx, cy))         
              _active_ring.set_radius(sigma * 1.15)                                                           
              prev_well = (cx, cy)                      
                                                                                                              
          frame += 1
          if frame % 3 == 0:                                                                                  
              _stick_dot.set_data([x], [y])             
              _spring_line.set_data([x, cx], [y, cy])
              _img_widget.value = _render()
              _info_label.value = (f'stick ({x:+.2f}, {y:+.2f})  '                                            
                                   f'→ well ({cx:+.2f}, {cy:+.2f})')
      except Exception as ex:                                                                                 
          _info_label.value = f'loop error: {ex}'; break
      time.sleep(0.02)                                                                                        
                                                        
_dyn_thread = threading.Thread(target=_pot_loop, daemon=True)                                                   
_dyn_thread.start()

# Using force directly

In [14]:
import io, threading, time, math
import numpy as np                                                                                              
import matplotlib
matplotlib.use('Agg')                                                                                           
import matplotlib.pyplot as plt                           
import matplotlib.patches as mpatches
import ipywidgets as w                                                                                          
from IPython.display import display
                                                                                                              
WELLS = [                                                 
  (-0.55, -0.45,  1.0,  0.22),
  ( 0.50, -0.30,  0.9,  0.20),                                                                                
  ( 0.05,  0.55,  1.0,  0.18),                                                                                
  (-0.20,  0.10,  0.6,  0.15),                                                                                
  ( 0.60,  0.42,  0.7,  0.20),                                                                                
]                                                                                                               
                                                        
def V(x, y):                                                                                                    
  v = np.zeros_like(np.asarray(x, float))               
  for cx, cy, A, s in WELLS:                                                                                  
      v -= A * np.exp(-((x - cx)**2 + (y - cy)**2) / (2 * s**2))                                              
  return v                                                                                                    
                                                                                                              
def grad_V(x, y):                                                                                               
  gx = gy = 0.0
  for cx, cy, A, s in WELLS:                                                                                  
      g = (A / s**2) * math.exp(-((x-cx)**2 + (y-cy)**2) / (2*s**2))
      gx += g * (x - cx)                                                                                      
      gy += g * (y - cy)                                                                                      
  return gx, gy                                                                                               
                                                                                                              
# ── Static background ─────────────────────────────────────────────────────────                                
_res = 250
_xg  = np.linspace(-1, 1, _res)                                                                                 
_yg  = np.linspace(-1, 1, _res)                                                                                 
_XX, _YY = np.meshgrid(_xg, _yg)
_ZZ = V(_XX, _YY)                                                                                               
                                                                                                              
fig, ax = plt.subplots(figsize=(5, 5), dpi=90)
ax.contourf(_XX, _YY, _ZZ, levels=50, cmap='RdYlBu_r')                                                          
ax.contour (_XX, _YY, _ZZ, levels=20, colors='k', linewidths=0.3, alpha=0.3)                                    
for cx, cy, A, s in WELLS:                                                                                      
  ax.add_patch(mpatches.Circle((cx, cy), s, color='white', fill=False,                                        
                                lw=1.2, ls='--', alpha=0.5, zorder=3))                                        
  ax.plot(cx, cy, 'w+', ms=11, mew=2, zorder=3)                                                               
ax.set_xlim(-1, 1); ax.set_ylim(-1, 1)                                                                          
ax.invert_yaxis()                                                                                               
ax.set_xlabel('X'); ax.set_ylabel('Y')                                                                          
ax.set_title('Potential field — constant force mode')                                                           
fig.tight_layout()
                                                                                                              
_bg_buf = io.BytesIO()                                    
fig.savefig(_bg_buf, format='png')                                                                              
_bg_bytes = _bg_buf.getvalue()                                                                                  

_stick_dot,  = ax.plot([0], [0], 'o', color='red',    ms=9, zorder=5)                                           
_force_line, = ax.plot([], [],        color='yellow', lw=2, zorder=4, alpha=0.85)
                                                                                                              
def _render():                                            
  buf = io.BytesIO()                                                                                          
  fig.savefig(buf, format='png')                        
  return buf.getvalue()
                                                                                                              
_img_widget = w.Image(value=_bg_bytes, format='png')
_info_label = w.Label('starting…')                                                                              
display(w.VBox([_img_widget, _info_label]))                                                                     

# ── Constant force FF loop ────────────────────────────────────────────────────                                
FORCE_SCALE = 220                                         
MAX_GRAD    = 3.0                                                                                               
ARROW_SCALE = 0.3                                         
                                                                                                              
if '_dyn_stop' in vars():
  _dyn_stop.set(); time.sleep(0.05)                                                                           
if '_pot_effect' in vars() and _pot_effect:                                                                     
  try: _pot_effect.free()
  except: pass                                                                                                
                                                        
_pot_effect = ff.alloc(ET_CONSTANT_FORCE)                                                                       
_pot_effect.set_base(direction_deg=0, gain=255,           
                   direction_enable=True, axis_x=True, axis_y=True)
_pot_effect.set_constant_force(0)                                                                               
_pot_effect.play()
                                                                                                              
_dyn_stop = threading.Event()                             

def _pot_loop():
  pos   = (0.0, 0.0)
  frame = 0                                                                                                   
  while not _dyn_stop.is_set():
      try:                                                                                                    
          result = ff.read_position()                   
          if result is not None:
              pos = result
          x, y = pos

          gx, gy = grad_V(x, y)                                                                               
          fx, fy = -gx, -gy      # device convention needs +∇V for attraction
                                                                                                              
          mag = math.sqrt(fx**2 + fy**2)                
          if mag < 1e-6:                                                                                      
              _pot_effect.set_constant_force(0)         
          else:
              clamped = int(min(mag / MAX_GRAD, 1.0) * FORCE_SCALE)
              angle   = (90.0 - math.degrees(math.atan2(fy, fx))) % 360                                       
              _pot_effect.set_base(direction_deg=angle, gain=255,                                             
                                   direction_enable=True,                                                     
                                   axis_x=True, axis_y=True)                                                  
              _pot_effect.set_constant_force(clamped)   
                                                                                                              
          frame += 1
          if frame % 3 == 0:                                                                                  
              _stick_dot.set_data([x], [y])             
              if mag > 1e-6:
                  sc = min(mag / MAX_GRAD, 1.0) * ARROW_SCALE
                  _force_line.set_data([x, x + fx/mag*sc],                                                    
                                       [y, y + fy/mag*sc])
              else:                                                                                           
                  _force_line.set_data([], [])          
              _img_widget.value = _render()                                                                   
              _info_label.value = (f'pos ({x:+.2f}, {y:+.2f})  '
                                   f'|F|={mag:.2f}  bearing={angle:.0f}°')                                    
      except Exception as ex:                                                                                 
          _info_label.value = f'loop error: {ex}'; break
      time.sleep(0.02)                                                                                        
                                                        
_dyn_thread = threading.Thread(target=_pot_loop, daemon=True)
_dyn_thread.start()

# Game

In [9]:
import io, threading, time, math
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import ipywidgets as w
from IPython.display import display as ipy_display

# ── Lennard-Jones parameters ──────────────────────────────────────────────────
EPS    = 1.0
SIG    = 0.13
RCUT   = 3.2 * SIG
R_EQ   = SIG * 2**(1/6)      # ≈ 0.146 — LJ equilibrium distance
BOX    = 0.90
N_FREE = 5
DT     = 0.006
TEMP   = 0.22
NU     = 0.015
MAX_SPEED = 4.0 * math.sqrt(TEMP)   # velocity cap — prevents simulation explosions

def _lj(dr):
    r2 = float(dr @ dr)
    if r2 < (0.75*SIG)**2 or r2 > RCUT**2: return np.zeros(2)
    ir2 = SIG**2 / r2;  ir6 = ir2**3
    return (4*EPS*(12*ir6**2 - 6*ir6) / r2) * dr

def _pe(dr):
    r2 = float(dr @ dr)
    if r2 < 1e-9 or r2 > RCUT**2: return 0.
    ir6 = (SIG**2/r2)**3
    return 4*EPS*(ir6**2 - ir6)

# ── Figure ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 5), dpi=88)
for obj in (fig, ax): obj.set_facecolor('#06060f')
fig.tight_layout(pad=0.4)
ax.set_xlim(-1,1); ax.set_ylim(-1,1); ax.set_aspect('equal'); ax.axis('off')
ax.set_title('⬡ Argon Herder — you are the red atom', color='#99bbff', fontsize=10)
ax.add_patch(plt.Rectangle((-BOX,-BOX), 2*BOX, 2*BOX,
    lw=1.6, edgecolor='#2255bb', fill=False, zorder=1))

# Equilibrium-distance ring around player (shows the "sweet spot")
_eq_ring = plt.Circle((0,0), R_EQ, color='#ff7755', fill=False,
                       lw=0.8, ls=':', alpha=0.5, zorder=2)
ax.add_patch(_eq_ring)
# Cutoff ring around player (shows interaction range)
_cut_ring = plt.Circle((0,0), RCUT, color='#334488', fill=False,
                        lw=0.5, ls='--', alpha=0.35, zorder=2)
ax.add_patch(_cut_ring)

_init_a   = np.linspace(0, 2*np.pi, N_FREE, endpoint=False)
_init_pos = np.column_stack([0.38*np.cos(_init_a), 0.38*np.sin(_init_a)])
_fh = [ax.add_patch(plt.Circle(_init_pos[i], SIG*0.68, color='#0033bb',
       alpha=0.22, zorder=2)) for i in range(N_FREE)]
_fc = [ax.add_patch(plt.Circle(_init_pos[i], SIG*0.42, color='#44aaff',
       alpha=0.88, zorder=3)) for i in range(N_FREE)]
_ph   = ax.add_patch(plt.Circle((0,0), SIG*0.68, color='#aa2200', alpha=0.28, zorder=2))
_pc   = ax.add_patch(plt.Circle((0,0), SIG*0.42, color='#ff3311', alpha=0.95, zorder=4))
_ptxt = ax.text(0, 0, 'you', color='white', fontsize=5.5, ha='center', va='center',
                fontweight='bold', zorder=5)

N_PAIRS = (N_FREE+1)*N_FREE//2
_bonds  = [ax.plot([],[],'-', lw=1.1, alpha=0, color='#88ccff', zorder=2)[0]
           for _ in range(N_PAIRS)]
_hud      = ax.text(-0.97, 0.93, '', color='#aaddff', fontsize=8.5,
                    family='monospace', va='top', zorder=6)
_best_txt = ax.text( 0.97, 0.93, '', color='#ffdd55', fontsize=8.5,
                    family='monospace', va='top', ha='right', zorder=6)
_haptic_txt = ax.text(0, -0.97, '', color='#88ff88', fontsize=7,
                      ha='center', va='bottom', family='monospace', zorder=6)

def _render():
    buf = io.BytesIO()
    fig.savefig(buf, format='png', facecolor='#06060f')
    return buf.getvalue()

_bg_buf = io.BytesIO()
fig.savefig(_bg_buf, format='png', facecolor='#06060f')
_img_w = w.Image(value=_bg_buf.getvalue(), format='png')
_lbl_w = w.Label('loading…')
ipy_display(w.VBox([_img_w, _lbl_w]))

# ── Force feedback — three effects ───────────────────────────────────────────
#
#  1. SPRING  — soft-clamped LJ force pulls stick toward equilibrium
#               uses tanh(F / F_KNEE) so large forces asymptote gracefully
#               instead of slamming to ±127
#
#  2. DAMPER  — light viscosity, makes motion deliberate
#
#  3. SINE    — encodes proximity haptic:
#               • silent  : no atom within RCUT
#               • 10 Hz / rising mag  : entering attraction zone (feel the draw)
#               • 6 Hz  / 50 mag      : near R_EQ sweet spot (bond throb)
#               • 30 Hz / rising mag  : inside repulsion zone (warn buzz)
#               • 55 Hz / 110 mag     : 0.4 s burst on new bond formed (confirmation)
#
FF_STIFF   = 40       # spring stiffness (reduced — tanh does the shaping)
FF_SAT     = 50      # spring saturation
FF_KNEE    = 6.0      # LJ force at which tanh reaches ~0.76 (gentle saturation)
FF_HSCALE  = 55       # max device units for the LJ haptic offset

if '_dyn_stop' in vars(): _dyn_stop.set(); time.sleep(0.08)
for _en in ('_ff_sp', '_ff_dm', '_ff_vib'):
    if _en in vars() and vars()[_en]:
        try: vars()[_en].free()
        except: pass

_ff_sp = ff.alloc(ET_SPRING)
_ff_sp.set_base(axis_x=True, axis_y=True, direction_enable=False, gain=255)
for _ax in (0,1):
    _ff_sp.set_condition(axis=_ax, cp_offset=0,
        pos_coeff=FF_STIFF, neg_coeff=FF_STIFF, pos_sat=FF_SAT, neg_sat=FF_SAT)
_ff_sp.play()

_ff_dm = ff.alloc(ET_DAMPER)
_ff_dm.set_base(axis_x=True, axis_y=True, direction_enable=False, gain=255)
for _ax in (0,1):
    _ff_dm.set_condition(axis=_ax, pos_coeff=50, neg_coeff=50, pos_sat=200, neg_sat=200)
_ff_dm.play()

_ff_vib = ff.alloc(ET_SINE)
_ff_vib.set_base(direction_deg=0, gain=220, direction_enable=True, axis_x=True, axis_y=True)
_ff_vib.set_periodic(magnitude=0, offset=0, phase_deg=0, period_ms=100)
_ff_vib.play()

# ── Game loop ─────────────────────────────────────────────────────────────────
_dyn_stop  = threading.Event()
_rng       = np.random.default_rng()

def _game():
    _a       = np.linspace(0, 2*np.pi, N_FREE, endpoint=False)
    free_pos = np.column_stack([0.38*np.cos(_a), 0.38*np.sin(_a)]).copy()
    free_vel = np.random.default_rng(3).normal(0, math.sqrt(TEMP), (N_FREE, 2))
    pp         = np.zeros(2)
    best_bonds = 0
    prev_bonds = 0
    buzz_until = 0.0   # time.time() until bond-confirmation buzz runs
    frame      = 0

    while not _dyn_stop.is_set():
        try:
            r = ff.read_position()
            if r is not None:
                pp = np.array([r[0], -r[1]])
            pp = np.clip(pp, -1, 1)

            # ── LJ forces ────────────────────────────────────────────────────
            F  = np.zeros((N_FREE, 2))
            Fp = np.zeros(2)

            for i in range(N_FREE):
                f = _lj(free_pos[i] - pp)
                F[i] = F[i] + f;  Fp = Fp - f
                for j in range(i+1, N_FREE):
                    f = _lj(free_pos[i] - free_pos[j])
                    F[i] = F[i] + f;  F[j] = F[j] - f

            # ── Euler + thermostat + speed cap ───────────────────────────────
            free_vel = free_vel + F * DT
            # Speed governor: rescale any atom exceeding MAX_SPEED
            speeds = np.linalg.norm(free_vel, axis=1, keepdims=True)
            too_fast = (speeds > MAX_SPEED).ravel()
            free_vel[too_fast] = (free_vel[too_fast]
                                  / speeds[too_fast] * MAX_SPEED)
            free_pos = free_pos + free_vel * DT

            kicks = _rng.random(N_FREE) < NU
            free_vel[kicks] = _rng.normal(0, math.sqrt(TEMP), (kicks.sum(), 2))

            for d in range(2):
                lo = free_pos[:,d] < -BOX;  hi = free_pos[:,d] > BOX
                free_pos[lo,d] = -BOX;  free_vel[lo,d] =  np.abs(free_vel[lo,d])
                free_pos[hi,d] =  BOX;  free_vel[hi,d] = -np.abs(free_vel[hi,d])

            # ── Score ────────────────────────────────────────────────────────
            all_pos = np.vstack([pp, free_pos])
            bonds = 0;  E_pot = 0.
            pair_info = []
            for i in range(N_FREE+1):
                for j in range(i+1, N_FREE+1):
                    dr     = all_pos[i] - all_pos[j]
                    rij    = np.linalg.norm(dr)
                    E_pot += _pe(dr)
                    bonded = rij < 1.55*R_EQ
                    if bonded: bonds += 1
                    pair_info.append((all_pos[i], all_pos[j], rij, bonded))
            best_bonds = max(best_bonds, bonds)

            # ── Proximity to nearest atom (for haptics) ───────────────────
            r_near = min(np.linalg.norm(free_pos[i] - pp) for i in range(N_FREE))

            # ── Haptic 1 : spring — soft tanh-clamped LJ guidance ────────
            # tanh maps raw force smoothly to ±1 so near-atom repulsion
            # becomes a firm push rather than a violent slam
            hx = math.tanh(Fp[0] / FF_KNEE) * FF_HSCALE
            hy = math.tanh(Fp[1] / FF_KNEE) * FF_HSCALE
            cpx = int(np.clip(pp[0]*127 + hx, -127, 127))
            cpy = int(np.clip(-pp[1]*127 - hy, -127, 127))
            _ff_sp.set_condition(axis=0, cp_offset=cpx,
                pos_coeff=FF_STIFF, neg_coeff=FF_STIFF, pos_sat=FF_SAT, neg_sat=FF_SAT)
            _ff_sp.set_condition(axis=1, cp_offset=cpy,
                pos_coeff=FF_STIFF, neg_coeff=FF_STIFF, pos_sat=FF_SAT, neg_sat=FF_SAT)

            # ── Haptic 2 : sine vibration — proximity grammar ─────────────
            now = time.time()
            if now < buzz_until:
                # Bond-formed confirmation buzz — 55 Hz sharp burst
                vib_period = 18;  vib_mag = 110
                haptic_label = '★ BOND!  55Hz buzz'
            elif r_near > RCUT:
                # Nothing nearby — silence
                vib_period = 100;  vib_mag = 0
                haptic_label = 'silent'
            elif r_near > R_EQ:
                # Attraction zone — slow draw sensation, 10 Hz, rising magnitude
                t = 1.0 - (r_near - R_EQ) / (RCUT - R_EQ)   # 0→1 as r→R_EQ
                vib_period = 100;  vib_mag = int(t * 38)
                haptic_label = f'attract  10Hz  mag={vib_mag}'
            elif r_near > 0.85 * R_EQ:
                # Right at the sweet spot — slow satisfying bond throb 6 Hz
                vib_period = 167;  vib_mag = 52
                haptic_label = 'sweet-spot  6Hz  throb'
            else:
                # Repulsion zone — rising buzz warns of core overlap
                t = 1.0 - (r_near - 0.75*SIG) / (R_EQ - 0.75*SIG)
                t = max(0., min(1., t))
                vib_period = int(100 - t * 67)   # 33–100 ms → 10–30 Hz
                vib_mag    = int(30 + t * 80)
                haptic_label = f'REPULSE  {1000//vib_period}Hz  mag={vib_mag}'

            _ff_vib.set_periodic(magnitude=vib_mag, offset=0,
                                 phase_deg=0, period_ms=vib_period)

            # Bond-formed event detection
            if bonds > prev_bonds:
                buzz_until = now + 0.40   # 400 ms confirmation buzz
            prev_bonds = bonds

            # ── Render (every 3rd step ≈ 15 fps) ─────────────────────────
            frame += 1
            if frame % 1 == 0:
                _pc.center = pp;  _ph.center = pp
                _eq_ring.center  = pp
                _cut_ring.center = pp
                _ptxt.set_position(pp)
                for i in range(N_FREE):
                    _fc[i].center = free_pos[i]
                    _fh[i].center = free_pos[i]

                for k, (p1, p2, rij, bonded) in enumerate(pair_info):
                    if rij < RCUT:
                        t = max(0., 1. - rij/RCUT)
                        _bonds[k].set_data([p1[0],p2[0]], [p1[1],p2[1]])
                        _bonds[k].set_alpha(t * 0.75)
                        _bonds[k].set_color('#ffdd88' if bonded else '#4488cc')
                    else:
                        _bonds[k].set_data([],[])

                T_now = np.sum(free_vel**2) / (2*N_FREE)
                max_b = N_FREE*(N_FREE+1)//2
                win   = ' ★ CRYSTAL!' if bonds == max_b else ''
                _hud.set_text(
                    f'bonds  {bonds}/{max_b}{win}\nEpot  {E_pot:+.2f}\nT  {T_now:.3f}')
                _best_txt.set_text(f'best\n{best_bonds}')
                _haptic_txt.set_text(f'haptic: {haptic_label}')

                _img_w.value = _render()
                _lbl_w.value = (f'bonds={bonds}/{max_b}  best={best_bonds}'
                                f'  r_near={r_near:.3f}  T={T_now:.3f}')

        except Exception as ex:
            _lbl_w.value = f'error: {ex}'; break
        time.sleep(0.02)

threading.Thread(target=_game, daemon=True, name='argon').start()
print('Game on!  Goal: maximise bonds.  ★ CRYSTAL = all 15 pairs bonded.')
print()
print('Haptic language:')
print('  silent              — no atoms within interaction range')
print('  slow 10 Hz draw     — atom in attraction zone, feel the pull')
print('  6 Hz throb          — right at bond equilibrium distance')
print('  fast 10–30 Hz buzz  — inside repulsion core, back off!')
print('  55 Hz burst 0.4 s   — new bond just formed!')

Game on!  Goal: maximise bonds.  ★ CRYSTAL = all 15 pairs bonded.

Haptic language:
  silent              — no atoms within interaction range
  slow 10 Hz draw     — atom in attraction zone, feel the pull
  6 Hz throb          — right at bond equilibrium distance
  fast 10–30 Hz buzz  — inside repulsion core, back off!
  55 Hz burst 0.4 s   — new bond just formed!


---
## Cleanup
Run this before shutting down the notebook.

In [56]:
for _h in [_cf, _pe, _sp, _dm, _fr, _in, _rp]:
    if _h[0]:
        try: _h[0].free()
        except: pass
        _h[0] = None
for e in _combo:
    try: e.free()
    except: pass
_combo.clear()

ff.close()
print("Device closed.")

NameError: name '_dm' is not defined